<a href="https://colab.research.google.com/github/LeandroHCarvalho/agentes-2026-2-equipe-agentes_especiais/blob/main/enc10_lab_colab_lara_frigulha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Encontro 10 — Integração, orquestração e memória

**Tópicos Especiais em IA · Agentes Inteligentes · IFES Serra · 2026/2**

Este é o lab do encontro, e ele roda **inteiro aqui no Colab**.

> **Mas repare numa coisa, porque é o assunto de hoje:** o código deste lab **não está
> neste notebook**. Ele está em **arquivos `.py`**, organizados num pacote chamado
> `vertice/`, e o que você vai rodar aqui são **comandos que executam esses arquivos**.
>
> É a primeira vez no semestre que isso acontece, e é de propósito. O projeto cresceu
> além do que uma célula sustenta.

A migração para o VS Code, com `clone`, `commit` e `push`, está no
**`GUIA_VSCODE.md`** e é tarefa para fazer com calma, fora da pressão da aula.

## 1 · Trazer o lab para cá

Duas formas. Se o professor publicou o repositório, cole a URL abaixo. Se não, deixe
vazio e o Colab vai pedir o arquivo `lab-enc10-vertice.zip`.

In [1]:
REPO_DO_LAB = ""     # <- cole aqui a URL do repositorio publico do lab, ou deixe vazio

import os, shutil, zipfile, pathlib

PASTA = "/content/lab-enc10"
shutil.rmtree(PASTA, ignore_errors=True)

if REPO_DO_LAB.strip():
    !git clone -q {REPO_DO_LAB} {PASTA}
else:
    from google.colab import files
    # Limpa arquivos zip antigos para evitar conflito de nomes no upload
    for item in pathlib.Path("/content").glob("lab-enc10-vertice*.zip"):
        try:
            os.remove(item)
        except Exception:
            pass

    print("Escolha o arquivo lab-enc10-vertice.zip")
    enviado = files.upload()
    nome = list(enviado)[0]
    os.makedirs(PASTA, exist_ok=True)
    with zipfile.ZipFile(nome) as z:
        z.extractall(PASTA)

# se o zip ou o repo criou uma pasta a mais, entra nela
if not pathlib.Path(PASTA, "testes.py").exists():
    for sub in pathlib.Path(PASTA).iterdir():
        if sub.is_dir() and (sub / "testes.py").exists():
            PASTA = str(sub)
            break

os.chdir(PASTA)
print("estou em:", os.getcwd())
print()
!find . -name "*.py" -not -path "*/.*" | sort

Escolha o arquivo lab-enc10-vertice.zip


Saving lab-enc10-vertice.zip to lab-enc10-vertice.zip
estou em: /content/lab-enc10

./rodar_memoria.py
./rodar_variancia.py
./testes.py
./vertice/artesanal.py
./vertice/casos.py
./vertice/config.py
./vertice/dominio.py
./vertice/framework.py
./vertice/__init__.py
./vertice/medicao.py


**Olhe a lista acima.** Isto é um **pacote Python**: uma pasta com `__init__.py` dentro,
e módulos que se importam entre si. Cada arquivo tem uma responsabilidade:

| Arquivo | Responsabilidade |
|---|---|
| `vertice/dominio.py` | o mundo sintético e as seis consultas |
| `vertice/casos.py` | os chamados e a verdade de terra |
| `vertice/config.py` | chave, modelo e preços |
| `vertice/artesanal.py` | o laço do Encontro 3 |
| `vertice/framework.py` | o mesmo agente no Agent Framework |
| `vertice/medicao.py` | consultas, tokens, dólares e a tabela |

**Era tudo isso que cabia numa célula até o Encontro 9.** Não cabe mais.

## 2 · Instalar

As versões fixadas do `requirements.txt`. **Não instale o meta-pacote `agent-framework`**:
ele arrasta 343 pacotes; os dois que importam fecham em 24.

Se o Colab pedir para reiniciar a sessão, reinicie e **rode a célula 1 de novo**.

In [2]:
%pip install -q -r requirements.txt
print("pronto. Se aparecer aviso de reiniciar a sessao, reinicie e volte a celula 1.")

pronto. Se aparecer aviso de reiniciar a sessao, reinicie e volte a celula 1.


## 3 · A chave

Continua vindo dos **Secrets do Colab**, como em todos os encontros desde o 2.

A célula lê o segredo e o coloca no **ambiente**, porque as células seguintes executam os
arquivos `.py` em outro processo, e esse processo não enxerga os *Secrets* direto.

**Nada é escrito em disco.** No VS Code, o mesmo módulo lê de um arquivo `.env` — e o
`config.py` procura nos dois lugares, sem mudar uma linha do agente.

In [3]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("chave carregada, termina em:", os.environ["GROQ_API_KEY"][-4:])
print()
print("Chave individual, nunca compartilhada: os limites do provedor sao por organizacao.")

chave carregada, termina em: Tzek

Chave individual, nunca compartilhada: os limites do provedor sao por organizacao.


## 4 · Os autotestes, que não gastam token

**Rode isto antes de qualquer coisa que gaste cota.** Leva menos de um segundo e pega o
erro mais caro do projeto: o mundo não conter a causa dos casos.

É o sucessor do `verificar_notebook.py` nesta fase: a mesma ideia, agora como script.

In [4]:
!python testes.py

1. O MUNDO CONTEM A CAUSA DOS TRES CASOS
  ok   C-103: a conta de diego.rocha existe
  ok   C-103: a conta esta ATIVA (o caso nao e de bloqueio)
  ok   C-103: diego.rocha NAO tem erp_leitura
  ok   C-106: a conta de elena.matos existe
  ok   C-106: a conta esta ATIVA (o caso nao e de bloqueio)
  ok   C-106: elena.matos NAO tem erp_leitura
  ok   C-108: a conta de joao.prado existe
  ok   C-108: a conta esta ATIVA (o caso nao e de bloqueio)
  ok   C-108: joao.prado NAO tem erp_leitura
  ok   a revisao de perfis de -6h esta no mundo
  ok   o servico esta NO AR (quem olhar so isso fecha sem causa)

2. A CAUSA NAO APARECE NAS DUAS PRIMEIRAS CONSULTAS
  ok   conta + servico NAO contem a causa
       e por isso que o C-103 precisa de uma TERCEIRA consulta.

3. ERRO QUE SE ANUNCIA (poka-yoke do retorno, Encontro 6)
  ok   usuario inexistente devolve ERRO
  ok   servico inexistente devolve ERRO
  ok   horas invalido devolve ERRO

4. O AVALIADOR, E O CONSERTO DA LIMITACAO 3 DO ENCONTRO 9
  ok  

## 5 · O teste de memória

Três chamados — **C-103, C-106 e C-108** — com **a mesma causa**: a revisão de perfis de
-6h removeu o grupo `erp_leitura` de 12 usuários, e os três estão entre eles.

Quem já apurou isso no primeiro não precisa apurar de novo. Quem não lembra, **refaz as
quatro consultas três vezes**.

A célula roda os três de dois jeitos, **sem memória** e **com memória**, e põe a conta
lado a lado. Leva uns 3 a 4 minutos, por causa das esperas de 20 s entre chamados.

In [5]:
!python rodar_memoria.py

TESTE DE MEMORIA -- Encontro 10
forma: laco artesanal

### SEM MEMORIA -- cada chamado do zero
  C-103 (diego.rocha)
    volta 1: status_do_servico({"servico":"erp-financeiro"})
    volta 2: consultar_conta({"usuario":"diego.rocha"})
    volta 3: mudancas_recentes({"horas":"24","servico":"erp-financeiro"})
  C-106 (elena.matos)
    volta 1: status_do_servico({"servico":"erp-financeiro"})
    volta 2: chamados_semelhantes({"servico":"erp-financeiro"})
  C-108 (joao.prado)
    volta 1: consultar_conta({"usuario":"joao.prado"})
    volta 2: status_do_servico({"servico":"erp-financeiro"})
    volta 3: chamados_semelhantes({"servico":"erp-financeiro"})

### COM MEMORIA -- a mesma conversa, do comeco ao fim
  C-103 (diego.rocha)
    volta 1: status_do_servico({"servico":"erp-financeiro"})
    volta 2: mudancas_recentes({"horas":"24","servico":"erp-financeiro"})
  C-106 (elena.matos)
    volta 1: status_do_servico({"servico":"erp-financeiro"})
    volta 2: chamados_semelhantes({"servico":"erp

### Leia a tabela antes de seguir

Três perguntas, e a célula já as imprime:

1. quantas consultas caíram?
2. quanto o custo caiu?
3. **a entrada cresceu quanto?**

**A terceira explica a segunda.** Memória é contexto carregado: cada chamado seguinte leva
a conversa inteira junto, então a entrada cresce. Se as consultas caírem muito e o custo
cair pouco, você acabou de medir **o preço da memória**.

E confira a coluna `causa` nas duas. **Economizar consulta só vale se a resposta continuar
certa.**

## 6 · O mesmo agente, agora no *framework*

Mesmo mundo, mesmas seis consultas, mesma instrução. **Só a orquestração muda.**

Se esta célula falhar, não é problema seu: siga com o resultado da anterior. O conteúdo do
bloco está lá.

In [6]:
!python rodar_memoria.py --framework

TESTE DE MEMORIA -- Encontro 10
forma: agent-framework

### SEM MEMORIA -- sem sessao, cada run isolado
  C-103 (diego.rocha)
    consulta 1: status_do_servico(servico='erp-financeiro')
    consulta 2: chamados_semelhantes(servico='erp-financeiro')
    consulta 3: mudancas_recentes(servico='erp-financeiro', horas='24')
    consultas: 3  tokens: 3363+926
  C-106 (elena.matos)
    consulta 4: status_do_servico(servico='erp-financeiro')
    consulta 5: chamados_semelhantes(servico='erp-financeiro')
    consultas: 2  tokens: 2397+745
  C-108 (joao.prado)
    consulta 6: consultar_conta(usuario='joao.prado')
    consulta 7: status_do_servico(servico='erp-financeiro')
    consulta 8: chamados_semelhantes(servico='erp-financeiro')
    consultas: 3  tokens: 3282+524

### COM MEMORIA -- uma sessao so, passada nos tres
  C-103 (diego.rocha)
    consulta 9: status_do_servico(servico='erp-financeiro')
    consulta 10: chamados_semelhantes(servico='erp-financeiro')
    consultas: 2  tokens: 2397+88

## 7 · O que sumiu, e é o ponto da aula

A célula abaixo mostra **o mesmo trecho** dos dois arquivos, lado a lado. Não gasta token.

In [7]:
import re, pathlib

def mostrar(caminho, titulo, inicio, linhas=18):
    texto = pathlib.Path(caminho).read_text(encoding="utf-8")
    i = texto.index(inicio)
    trecho = texto[i:].splitlines()[:linhas]
    print("=" * 74)
    print(titulo)
    print("=" * 74)
    print("\n".join(trecho))
    print()

mostrar("vertice/artesanal.py",
        "ARTESANAL -- declarar UMA ferramenta", "_D_USUARIO =", 12)
mostrar("vertice/framework.py",
        "FRAMEWORK -- declarar AS SEIS", "FERRAMENTAS = [", 10)

print("=" * 74)
print("A ferramenta virou a FUNCAO, e a description virou a DOCSTRING.")
print("Todo o criterio da clinica do Encontro 4 continua valendo, no mesmo")
print("lugar de sempre. O framework tira de voce o SCHEMA, nao a decisao.")
print()
print("E o laco de 30 linhas do Encontro 3 virou uma chamada:")
print("    agente.run(pedido, session=sessao)")
print()
print("E a mesma troca que voces mediram no Encontro 9 com o LangChain:")
print("menos linhas, e as decisoes escondidas atras de padroes.")

ARTESANAL -- declarar UMA ferramenta
_D_USUARIO = "identificador do usuario, no formato nome.sobrenome"
_D_SERVICO = "nome do servico. Um de: portal-rh, erp-financeiro, vpn, email"

FERRAMENTAS = [
    _f("consultar_conta",
       "Devolve o estado da conta de um usuario: ativa ou bloqueada, os grupos de "
       "acesso que ela tem, e ha quanto tempo a senha foi trocada. Nao diz nada "
       "sobre o servico: para isso use status_do_servico.",
       _p("usuario", _D_USUARIO)),
    _f("status_do_servico",
       "Devolve o estado ATUAL de um servico corporativo e desde quando. Nao diz o "
       "que mudou nele: para isso use mudancas_recentes.",

FRAMEWORK -- declarar AS SEIS
FERRAMENTAS = [
    _contado(dominio.consultar_conta),
    _contado(dominio.status_do_servico),
    _contado(dominio.mudancas_recentes),
    _contado(dominio.chamados_semelhantes),
    _contado(dominio.historico_do_usuario),
    _contado(dominio.base_de_conhecimento),
]



A ferramenta virou a FUNCAO, e a descr

## 8 · A tarefa: sair do Colab

Este notebook rodou tudo. O que ele **não** faz é `commit` e `push`, e isso você vai
precisar antes do Marco 1, que é entregue por ***tag*** no repositório.

**O passo a passo completo está no `GUIA_VSCODE.md`**, escrito para ser seguido sem
pressa: instalar, autenticar, clonar, rodar e empurrar. Leva cerca de 40 minutos na
primeira vez, e o plantão existe para isso.

> **Por que vale a pena, e não é burocracia:** a partir daqui o projeto da sua equipe é um
> repositório, não uma pasta de notebooks. É o que torna possível duas pessoas mexerem no
> mesmo código, e é o que o Marco 2 vai pedir com *branch* e *pull request*.